In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
!pip install biopython
from Bio.PDB import PDBParser
from torch.utils.data import Dataset, DataLoader
import sys
sys.path.append('scripts')

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import pdb_to_graph
import mldft_surrogate
import verify_twin_pipeline
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper

In [37]:
protein = "proteins/1ACX.pdb"
tensor = pdb_voxelizier.pdb_to_tensor(protein,grid_size=32)
model = cnn_mlp_encoder.ProteinPhysicsEncoder(num_sites=4)

In [38]:
tensor = torch.tensor(tensor,dtype=torch.float32)
print(tensor.shape)

torch.Size([1, 1, 32, 32, 32])


In [39]:
model.load_state_dict(torch.load(r"\Users\Aaliyah\OneDrive\Documents\GitHub\Learning-Quantum-Hamiltonians-from-Protein-Structure\protein_cnn2.pth"))
model.eval()
with torch.no_grad():
    cnn_coefficients = model(tensor)
cnn_coefficients = (cnn_coefficients.cpu().numpy()[0])
print(cnn_coefficients)

[-0.07580934 -0.07300865  0.10659735 -0.07843549 -0.12415397  0.19897449
 -0.01125928  0.27920228  0.33274865 -0.22443448]


In [40]:
# 1. Run Track B (ML-DFT)
graph_data = pdb_to_graph.pdb_to_graph(protein, distance_threshold=5.0)
mldft_coefficients = mldft_surrogate.get_mldft_hamiltonian(graph_data, num_qubits=4)

In [41]:
print(mldft_coefficients)

[-0.01578134  0.07778591 -0.23263177 -0.39413854 -0.33330688 -0.07920618
  0.14350687  0.0137698  -0.46862298  0.04051202]


In [42]:
# 2. Run the Verification (Assuming you have 'cnn_coefficients' from Track A)
# Use dummy data here just to test the script if needed:
#cnn_coefficients = mldft_coefficients + np.random.normal(0, 0.01, 10)
verify_twin_pipeline.cross_verify_pipelines(cnn_coefficients, mldft_coefficients, num_sites=4)

=== TWIN PIPELINE VERIFICATION REPORT ===

[Checkpoint 1] Coefficient Mean Absolute Error (MAE): 0.283961 eV
-> Status: WARNING (Check spatial mapping drift)

[Checkpoint 2] Physical Ground State Energy (E0)
Track A (3D CNN) E0 : -0.640806 eV
Track B (ML-DFT) E0 : -0.690443 eV
Delta E (Error)     : 0.049637 eV
-> Status: FAIL (Exceeds Chemical Accuracy. Do not send to QPU.)
